<a href="https://colab.research.google.com/github/jamesymwang/Kp-predict_MACCS-and-Molecular-Transformer/blob/main/openpom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Environment Diagnostics
!nvcc --version
!python --version
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"Is CUDA available? {torch.cuda.is_available()}")

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
Python 3.12.12
PyTorch Version: 2.9.0+cu126
CUDA Version: 12.6
Is CUDA available? True


In [1]:
# Cell 1: Install Exact Dependencies for OpenPOM
# 1. 安装 RDKit（化学后端）
!pip install rdkit

# 2. 安装 DeepChem（预发布版，以确保 Colab 兼容性）
!pip install --pre deepchem

# 3. 将 PyTorch 降级至 2.4.0（DGL 兼容性所需）
# 我们强制安装支持 CUDA 12.4 的版本 2.4.0
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu124

# 4. 安装与 PyTorch 2.4 匹配的 DGL（深度图库）
!pip install dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html

# 5. 安装 OpenPOM 和 DGL-LifeSci
!pip install openpom dgllife

# 6. 验证安装
import torch
import dgl
print(f"PyTorch 版本: {torch.__version__} (应为 2.4.0+cu124)")
print(f"DGL 版本: {dgl.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 46.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 32.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 79.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.4/883.4 kB 60.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 131.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 816.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.4/128.4 MB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207

DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)
PyTorch 版本: 2.4.0+cu124 (应为 2.4.0+cu124)
DGL 版本: 2.4.0+cu124
CUDA 可用: True


In [2]:
# Cell 2: Download Dataset
import os

# 从 OpenPOM 仓库下载整理好的数据集
!wget -N https://raw.githubusercontent.com/ARY2260/openpom/main/openpom/data/curated_datasets/curated_GS_LF_merged_4983.csv

input_file = "curated_GS_LF_merged_4983.csv"
print("数据集已下载。")

--2025-12-07 12:36:30--  https://raw.githubusercontent.com/ARY2260/openpom/main/openpom/data/curated_datasets/curated_GS_LF_merged_4983.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1647971 (1.6M) [text/plain]
Saving to: ‘curated_GS_LF_merged_4983.csv’

curated_GS_LF_merge 100%[===================>]   1.57M  --.-KB/s    in 0.05s   

Last-modified header missing -- time-stamps turned off.
2025-12-07 12:36:30 (28.7 MB/s) - ‘curated_GS_LF_merged_4983.csv’ saved [1647971/1647971]

数据集已下载。


In [6]:
# Cell 3: Extract Embeddings (Fixed)
import deepchem as dc
import pandas as pd
import numpy as np
import torch
from openpom.feat.graph_featurizer import GraphFeaturizer, GraphConvConstants
from openpom.models.mpnn_pom import MPNNPOMModel

# 1. Load Data
input_file = "curated_GS_LF_merged_4983.csv"
df_gslf = pd.read_csv(input_file)

# --- FIX: Exclude 'descriptors' text column ---
# We must filter out 'descriptors' because it contains text (e.g., 'fishy')
# that causes the "could not convert string to float" error.
ignore_cols = ['nonStereoSMILES', 'descriptors', 'cid', 'mol_id', 'smiles', 'canonical_smiles']
TASKS = [c for c in df_gslf.columns if c not in ignore_cols]
print(f"Identified {len(TASKS)} binary odor tasks (excluding text columns).")

# 2. Featurize (Convert Text to Graphs)
print("Featurizing molecules... (This takes ~1 minute)")
featurizer = GraphFeaturizer()
loader = dc.data.CSVLoader(tasks=TASKS, feature_field='nonStereoSMILES', featurizer=featurizer)
dataset = loader.create_dataset(inputs=[input_file])

# 3. Calculate Imbalance Ratios (Manual Calculation)
# This creates the necessary weights for the model structure without crashing
print("Calculating class imbalance ratios...")
y = dataset.y
train_ratios = []
# If for some reason y is empty, we provide a default
if y.shape[1] == 0:
    train_ratios = [0.0] * len(TASKS)
else:
    for i in range(y.shape[1]):
        n_pos = np.sum(y[:, i] == 1)
        n_neg = np.sum(y[:, i] == 0)
        if n_pos == 0:
            ratio = 0.0
        else:
            ratio = float(n_neg) / float(n_pos)
        train_ratios.append(ratio)

# 4. Initialize Model
# We set up the model architecture required to generate the 256-dim embeddings
n_tasks = len(TASKS) if len(TASKS) > 0 else 1

model = MPNNPOMModel(
    n_tasks=n_tasks,
    batch_size=128,
    learning_rate=0.001,
    class_imbalance_ratio=train_ratios,
    loss_aggr_type='sum',
    node_out_feats=100,
    edge_hidden_feats=75,
    edge_out_feats=100,
    num_step_message_passing=5,
    mpnn_residual=True,
    message_aggregator_type='sum',
    mode='classification',
    number_atom_features=GraphConvConstants.ATOM_FDIM,
    number_bond_features=GraphConvConstants.BOND_FDIM,
    n_classes=1,
    readout_type='set2set',
    num_step_set2set=3,
    num_layer_set2set=2,
    ffn_hidden_list=[392, 392],
    ffn_embeddings=256,         # This ensures the output embeddings are 256 dimensions
    ffn_activation='relu',
    ffn_dropout_p=0.12
)

# 5. Extract and Save Embeddings
print("Extracting embeddings... (This might take 1-2 minutes)")
embeddings = model.predict_embedding(dataset)

# Save to CSV
df_embeddings = pd.DataFrame(embeddings)
# Map back to CIDs if available, otherwise just save
if 'cid' in df_gslf.columns:
    df_embeddings['cid'] = df_gslf['cid'].values

output_filename = "openpom_embeddings.csv"
df_embeddings.to_csv(output_filename, index=False)

print(f"Success! Extracted embeddings shape: {embeddings.shape}")
print(f"Saved to '{output_filename}'")

Identified 138 binary odor tasks (excluding text columns).
Featurizing molecules... (This takes ~1 minute)
Calculating class imbalance ratios...
Extracting embeddings... (This might take 1-2 minutes)
Success! Extracted embeddings shape: (4983, 256)
Saved to 'openpom_embeddings.csv'


In [7]:
# Cell 4: Extract Embeddings for New Data (Stimulus Mapping)
import numpy as np
import pandas as pd
import deepchem as dc

# 1. Load the New Data
new_input_file = "stimulus_smiles_mapping.csv"
df_new = pd.read_csv(new_input_file)

print(f"Loaded new file with {len(df_new)} molecules.")

# 2. Featurize the New SMILES
# Note: We change 'feature_field' to 'SMILES' to match your new CSV header
# We pass tasks=[] because this new file has no labels, we only want predictions.
print("Featurizing new molecules...")
loader_new = dc.data.CSVLoader(tasks=[], feature_field='SMILES', featurizer=featurizer)
dataset_new = loader_new.create_dataset(inputs=[new_input_file])

# 3. Extract Embeddings
print("Generating embeddings for new dataset...")
# We reuse the 'model' object that is already active in your session
new_embeddings = model.predict_embedding(dataset_new)

# 4. Save as .npy file
output_npy = "stimulus_embeddings.npy"
np.save(output_npy, new_embeddings)

print(f"Success! New embeddings shape: {new_embeddings.shape}")
print(f"Saved to '{output_npy}'")

# Optional: Verify by checking the first few values
print("First embedding sample:", new_embeddings[0][:10])

Loaded new file with 196 molecules.
Featurizing new molecules...
Generating embeddings for new dataset...
Success! New embeddings shape: (196, 256)
Saved to 'stimulus_embeddings.npy'
First embedding sample: [-0.04044907  0.05958064  0.01359227  0.00582833  0.00991368 -0.02056484
  0.00845871 -0.07043297  0.02232944  0.01343226]
